# Uniform Space-Filling in 2D

This notebook demonstrates a basic sequential space-filling workflow in a 2D input space.

- A **candidate set** defines the allowable input combinations.
- A **design** is a subset of those candidate points.
- **Minimax** and **Maximin** are two optimality modes for sampling points selection. 
The workflow has two stages:

- **USF-1:** build minimax and maximin designs of size 8, 9, and 10.
- **USF-2:** treat the chosen 8-run minimax design as completed work and add new runs.

The notebook uses the following API calls: `load_csv`, `prepare_design_setup`, `estimate_uniform_runtime`, `design_uniform_batch`, and `plot_pair_matrix`. Files created during the run are saved under `scripts/output/nb_output`.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    elif (ROOT.parent.parent / "src").exists():
        ROOT = ROOT.parent.parent
    else:
        raise RuntimeError("Could not locate the project root.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from idaes_sdoe import ColumnRoles, load_csv, prepare_design_setup, write_csv
from idaes_sdoe.design import design_uniform_batch, estimate_uniform_runtime
from idaes_sdoe.plotting import plot_pair_matrix, suggest_histogram_bin_map, write_figure

DATA_FILE = ROOT / "examples" / "supporting_data" / "SDOE_Ex1_Candidates.csv"
DESIGN_SIZES = [8, 9, 10]
USF2_ADDITIONAL_SIZES = [1, 2, 3]
NUM_RESTARTS = 100_000
CALIBRATION_RESTARTS = 100
PLOT_COLUMNS = ["X1", "X2"]
HISTOGRAM_TARGET_BINS = 40

def make_output_dir(label: str) -> Path:
    stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    path = ROOT / "examples" / "temp"/ "nb_output" / f"{stamp}_{label}"
    path.mkdir(parents=True, exist_ok=True)
    return path

def write_json(payload: dict[str, object], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)

def save_figure(figure, path: Path) -> None:
    try:
        write_figure(figure, path)
    except Exception as exc:
        print(f"warning: could not save {path.name}: {exc}")

output_dir = make_output_dir("example_uniform_nb")
output_dir

## 1. Load and inspect the candidate set

The candidate set is the search space for the design algorithm. In this example it is a regular grid in `X1` and `X2`. The first step is to load the table, define which columns are design inputs, inspect the bounds, and view the pairwise plot of the full candidate region.

In [ ]:
candidate = load_csv(DATA_FILE)
setup = prepare_design_setup(
    candidate=candidate,
    roles=ColumnRoles(inputs=PLOT_COLUMNS),
    auto_index=True,
    index_column="_id",
)

write_csv(setup.candidate, output_dir / "candidate_set.csv")

display(Markdown(f"**Rows:** {len(setup.candidate)}  \n**Columns:** {', '.join(setup.candidate.columns)}"))
display(setup.candidate.head())
display(setup.candidate[PLOT_COLUMNS].agg(["min", "max"]))

candidate_bins = suggest_histogram_bin_map(
    setup.candidate[PLOT_COLUMNS],
    target_bins=HISTOGRAM_TARGET_BINS,
)
candidate_fig = plot_pair_matrix(
    setup.candidate[PLOT_COLUMNS],
    columns=PLOT_COLUMNS,
    title="Candidate set pairwise plot",
    histogram_bins=candidate_bins,
)
save_figure(candidate_fig, output_dir / "candidate_pairwise.pdf")
candidate_fig

## 2. Run USF-1

This stage constructs several alternative designs from the same candidate set. Each run of `design_uniform_batch` searches over many random subsets and keeps the best one found for the chosen criterion. The result is a small family of designs that can be compared by size, criterion value, and point placement.

In [ ]:
stage1_rows = []
stage1_results: dict[str, dict[int, object]] = {}

# Two optimality modes- minimax and maximin - are examined below
for mode in ("minimax", "maximin"):
    mode_dir = output_dir / "usf1" / mode
    design_dir = mode_dir / "designs"
    plot_dir = mode_dir / "plots"
    design_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    estimate = estimate_uniform_runtime(
        setup=setup,
        design_sizes=DESIGN_SIZES,
        target_restarts=NUM_RESTARTS,
        mode=mode,
        calibration_restarts=CALIBRATION_RESTARTS,
    )
    write_json(estimate, mode_dir / "runtime_estimate.json")

    started = time.perf_counter()
    results = design_uniform_batch(
        setup=setup,
        design_sizes=DESIGN_SIZES,
        num_restarts=NUM_RESTARTS,
        mode=mode,
    )
    stage1_results[mode] = {result.design_size: result for result in results}
    elapsed = time.perf_counter() - started

    for result in results:
        design_file = design_dir / f"{mode}_design_{result.design_size:02d}.csv"
        pair_file = plot_dir / f"{mode}_design_{result.design_size:02d}_pairwise.pdf"
        write_csv(result.design, design_file)

        bins = suggest_histogram_bin_map(
            result.design[PLOT_COLUMNS],
            reference=setup.candidate[PLOT_COLUMNS],
            target_bins=HISTOGRAM_TARGET_BINS,
        )
        fig = plot_pair_matrix(
            result.design[PLOT_COLUMNS],
            columns=PLOT_COLUMNS,
            candidate=setup.candidate[PLOT_COLUMNS],
            title=(
                f"{mode.title()} design, size {result.design_size}, "
                f"criterion={result.criterion_value:.6f}"
            ),
            histogram_bins=bins,
        )
        save_figure(fig, pair_file)

        stage1_rows.append(
            {
                "mode": mode,
                "design_size": result.design_size,
                "criterion_value": result.criterion_value,
                "elapsed_time": result.elapsed_time,
                "design_file": str(design_file),
                "pair_plot_pdf": str(pair_file),
            }
        )

    write_json(
        {
            "mode": mode,
            "design_sizes": DESIGN_SIZES,
            "num_restarts": NUM_RESTARTS,
            "calibration_restarts": CALIBRATION_RESTARTS,
            "elapsed_seconds": elapsed,
        },
        mode_dir / "run_summary.json",
    )

stage1_summary = pd.DataFrame(stage1_rows)
write_csv(stage1_summary, output_dir / "usf1_created_designs_summary.csv")
stage1_summary[["mode", "design_size", "criterion_value", "elapsed_time"]]

In [ ]:
display(Markdown("**USF-1 design plots**"))
for mode in ("minimax", "maximin"):
    for size in DESIGN_SIZES:
        result = stage1_results[mode][size]
        bins = suggest_histogram_bin_map(
            result.design[PLOT_COLUMNS],
            reference=setup.candidate[PLOT_COLUMNS],
            target_bins=HISTOGRAM_TARGET_BINS,
        )
        fig = plot_pair_matrix(
            result.design[PLOT_COLUMNS],
            columns=PLOT_COLUMNS,
            candidate=setup.candidate[PLOT_COLUMNS],
            title=(
                f"{mode.title()} design, size {size}, "
                f"criterion={result.criterion_value:.6f}"
            ),
            histogram_bins=bins,
        )
        display(fig)


## 3. Run USF-2

Sequential design starts from runs that already exist. Here the selected 8-run minimax design is treated as **previous data**, and the new design is chosen to complement those existing points. In this notebook, points that were already run are removed from the new candidate pool so the augmentation adds only new runs.

In [ ]:
previous_design = stage1_results["minimax"][8].design.copy()
remaining_candidate = setup.candidate.loc[~setup.candidate["_id"].isin(previous_design["_id"])].copy()

sequential_setup = prepare_design_setup(
    candidate=remaining_candidate,
    previous=previous_design,
    roles=ColumnRoles(index="_id", inputs=PLOT_COLUMNS),
    auto_index=False,
)

write_csv(sequential_setup.candidate, output_dir / "usf2_candidate_remaining.csv")
write_csv(sequential_setup.previous, output_dir / "usf2_previous_data.csv")

setup_fig = plot_pair_matrix(
    sequential_setup.candidate[PLOT_COLUMNS],
    columns=PLOT_COLUMNS,
    previous=sequential_setup.previous[PLOT_COLUMNS],
    title="USF-2 setup: remaining candidates and previous data",
    histogram_bins=suggest_histogram_bin_map(
        sequential_setup.candidate[PLOT_COLUMNS],
        reference=pd.concat(
            (sequential_setup.candidate[PLOT_COLUMNS], sequential_setup.previous[PLOT_COLUMNS]),
            ignore_index=True,
        ),
        target_bins=HISTOGRAM_TARGET_BINS,
    ),
)
save_figure(setup_fig, output_dir / "usf2_setup_pairwise.pdf")
setup_fig

In [ ]:
stage2_rows = []
stage2_results: dict[str, dict[int, object]] = {}

for mode in ("minimax", "maximin"):
    mode_dir = output_dir / "usf2" / mode
    design_dir = mode_dir / "designs"
    plot_dir = mode_dir / "plots"
    design_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    estimate = estimate_uniform_runtime(
        setup=sequential_setup,
        design_sizes=USF2_ADDITIONAL_SIZES,
        target_restarts=NUM_RESTARTS,
        mode=mode,
        calibration_restarts=CALIBRATION_RESTARTS,
    )
    write_json(estimate, mode_dir / "runtime_estimate.json")

    results = design_uniform_batch(
        setup=sequential_setup,
        design_sizes=USF2_ADDITIONAL_SIZES,
        num_restarts=NUM_RESTARTS,
        mode=mode,
    )
    stage2_results[mode] = {result.design_size: result for result in results}

    for result in results:
        total_size = len(sequential_setup.previous) + result.design_size
        additional_file = design_dir / f"{mode}_additional_{result.design_size:02d}.csv"
        combined_file = design_dir / f"{mode}_combined_total_{total_size:02d}.csv"
        pair_file = plot_dir / f"{mode}_additional_{result.design_size:02d}_pairwise.pdf"
        combined = pd.concat((sequential_setup.previous, result.design), ignore_index=True)
        write_csv(result.design, additional_file)
        write_csv(combined, combined_file)

        bins = suggest_histogram_bin_map(
            result.design[PLOT_COLUMNS],
            reference=sequential_setup.candidate[PLOT_COLUMNS],
            target_bins=HISTOGRAM_TARGET_BINS,
        )
        fig = plot_pair_matrix(
            result.design[PLOT_COLUMNS],
            columns=PLOT_COLUMNS,
            candidate=sequential_setup.candidate[PLOT_COLUMNS],
            previous=sequential_setup.previous[PLOT_COLUMNS],
            title=(
                f"USF-2 {mode.title()} augmentation, +{result.design_size} "
                f"(total {total_size}), criterion={result.criterion_value:.6f}"
            ),
            histogram_bins=bins,
        )
        save_figure(fig, pair_file)

        stage2_rows.append(
            {
                "mode": mode,
                "additional_design_size": result.design_size,
                "total_design_size": total_size,
                "criterion_value": result.criterion_value,
                "elapsed_time": result.elapsed_time,
                "additional_design_file": str(additional_file),
                "combined_design_file": str(combined_file),
                "pair_plot_pdf": str(pair_file),
            }
        )

stage2_summary = pd.DataFrame(stage2_rows)
write_csv(stage2_summary, output_dir / "usf2_created_designs_summary.csv")
stage2_summary[["mode", "additional_design_size", "total_design_size", "criterion_value", "elapsed_time"]]

In [ ]:
display(Markdown("**USF-2 augmentation plots**"))
for mode in ("minimax", "maximin"):
    for size in USF2_ADDITIONAL_SIZES:
        result = stage2_results[mode][size]
        total_size = len(sequential_setup.previous) + result.design_size
        bins = suggest_histogram_bin_map(
            result.design[PLOT_COLUMNS],
            reference=sequential_setup.candidate[PLOT_COLUMNS],
            target_bins=HISTOGRAM_TARGET_BINS,
        )
        fig = plot_pair_matrix(
            result.design[PLOT_COLUMNS],
            columns=PLOT_COLUMNS,
            candidate=sequential_setup.candidate[PLOT_COLUMNS],
            previous=sequential_setup.previous[PLOT_COLUMNS],
            title=(
                f"USF-2 {mode.title()} augmentation, +{size} "
                f"(total {total_size}), criterion={result.criterion_value:.6f}"
            ),
            histogram_bins=bins,
        )
        display(fig)
